# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library. This dataset contains survey data on mental health indicators among residents of Kilifi County, including demographics and psychological assessments (GAD-7, PHQ-9, PSQ).

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Access dataset metadata as an object, print key information
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Citation: {getattr(metadata, 'citeAs', None)}")
print(f"Keywords: {getattr(metadata, 'keywords', None)}")

## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers.

In Croissant datasets, a "record set" is a logical collection of records (like a main table or sheet). Let's list available record sets and the fields for each. We will reference all by their `@id`.

In [ ]:
# List all record sets by their @id
print("Available Record Sets (by @id):")
for rs in dataset.record_sets:
    print(f"  @id: {rs.id}  |  name: {getattr(rs, 'name', None)}  |  description: {getattr(rs, 'description', None)}")

# For each record set, list all fields and columns by their @id
for rs in dataset.record_sets:
    print(f"\nRecord set @id: {rs.id} fields and columns:")
    if hasattr(rs, 'fields') and rs.fields:
        for field in rs.fields:
            print(f"  Field @id: {field.id}  |  data type: {getattr(field, 'data_type', None)}  |  name: {getattr(field, 'name', None)}")
            # If field has associated columns, list them
            if hasattr(field, 'columns') and field.columns:
                for col in field.columns:
                    print(f"    Column @id: {col.id}  |  name: {getattr(col, 'name', None)}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. We reference record sets and field `@id`s identified above.

In [ ]:
# Identify the @id values for main record set(s) from the printed list above.
# For this dataset, it is common that the main survey table is the primary record set.

# Let's retrieve all record set @ids programmatically and use them for DataFrame extraction.
record_sets = [rs.id for rs in dataset.record_sets]
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for Record Set @id: {record_set_id}, shape: {dataframes[record_set_id].shape}")

# For illustration, display columns and a preview for the main record set (first one)
if dataframes:
    primary_rs = list(dataframes.keys())[0]
    print(f"\nColumns for record set {primary_rs}:")
    print(dataframes[primary_rs].columns.tolist())
    display(dataframes[primary_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates removing outliers, transforming data, or grouping data for summary analysis.

Let's pick a known mental health assessment score (e.g., GAD-7, PHQ-9, or PSQ) for numeric analysis. Be sure to use the field/column `@id`.

In [ ]:
# Select main DataFrame; refer by record set @id
main_rs_id = primary_rs  # This is the @id that was printed above

# Let's see which columns exist
df = dataframes[main_rs_id]
print(df.columns)

# For demonstration, search for possible GAD-7, PHQ-9, or PSQ score field/column @id
# Replace these with the actual @id from the previous cell's output if different
# Here we guess that 'gad_7_total_score', 'phq_9_total_score', 'psq_total_score' are column names or ids
possible_score_fields = [col for col in df.columns if 'gad' in col.lower() or 'phq' in col.lower() or 'psq' in col.lower()]
print(f"Possible numeric score fields found: {possible_score_fields}")

# Pick 'gad_7_total_score' if available, else fallback
if 'gad_7_total_score' in df.columns:
    numeric_field = 'gad_7_total_score'  # This should match column's @id if available
elif 'phq_9_total_score' in df.columns:
    numeric_field = 'phq_9_total_score'
elif 'psq_total_score' in df.columns:
    numeric_field = 'psq_total_score'
else:
    # Fallback: just use the first available possible score column
    numeric_field = possible_score_fields[0] if possible_score_fields else None

if numeric_field is None:
    raise ValueError("Could not find a numeric mental health score column.")

# Filter records with scores > threshold (e.g., GAD-7 score > 10)
threshold = 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)} found.")
display(filtered_df[[numeric_field]].head())

# Normalize the score field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a relevant demographic field, e.g. 'gender' or 'village'.
possible_group_fields = [col for col in df.columns if col.lower() in ['gender', 'village', 'level_of_education']]
if possible_group_fields:
    group_field = possible_group_fields[0]
    print(f"Grouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(grouped_df.head())
else:
    print("No demographic group field (gender/village/education) found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the chosen numeric field
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field], bins=10, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If we grouped by a group_field above, show a boxplot
if 'group_field' in locals() and group_field in df.columns:
    plt.figure(figsize=(8,5))
    sns.boxplot(data=df, x=group_field, y=numeric_field)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

# Pairplot for available score fields if at least 2
if len(possible_score_fields) >= 2:
    sns.pairplot(df[possible_score_fields].dropna())
    plt.suptitle("Pairwise Relationships of Mental Health Scores", y=1.02)
    plt.show()

## 6. Conclusion
We have demonstrated loading and exploring the FAIR² Mental Health Survey Dataset using `mlcroissant`:
- Loaded Croissant-compliant metadata and data records using record set and field `@id`s
- Identified fields for psychological assessment scores, filtered and normalized scores
- Visualized distributions and group differences by demographic fields

This reproducible workflow can be adapted to a range of Croissant schema datasets for FAIR, AI-ready analytics in mental health research and beyond.